# 05 · Purchase Decision Driver Analysis

**Research question:** When consumers write reviews for anti-spot products, what drives their purchase decision — ingredients, brand, authority (dermatologists / social media), price, or results?

**Sub-question A:** Distribution of decision signals in review text for anti-spot vs. other skincare vs. makeup.

**Hypotheses:**
- Anti-spot products have higher ingredient-driven mentions than general skincare ("ingredient-savvy" consumers)
- Authority-driven signals (dermatologist / Reddit / TikTok) are higher in anti-spot than makeup

In [ ]:
import pandas as pd
import numpy as np
import re
import glob
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 120)

## 1. Load Data

In [ ]:
# Load all review files
review_files = sorted(glob.glob('../data/raw/reviews_*.csv'))
rev = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in review_files],
    ignore_index=True
)
print(f'Total reviews: {len(rev):,}')

# Load product metadata
prod = pd.read_csv('../data/raw/product_info.csv', low_memory=False)
print(f'Total products: {len(prod):,}')

# Merge category info
df = rev.merge(
    prod[['product_id', 'primary_category', 'secondary_category', 'highlights', 'price_usd', 'brand_name']],
    on='product_id', how='left',
    suffixes=('_review', '_product')
)
df['review_text'] = df['review_text'].fillna('')
print(f'Merged shape: {df.shape}')

## 2. Define Product Segments

Three comparison groups:
- **Anti-spot** — brightening, dark spot, hyperpigmentation products
- **Other skincare** — moisturizers, cleansers, sunscreen, etc.
- **Makeup** — foundation, concealer, blush, etc.

In [ ]:
# Anti-spot: keyword match on product name OR highlights tag
ANTISPOT_KEYWORDS = (
    r'bright|dark.?spot|spot.?correct|hyperpigment|even.?skin|vitamin.?c serum|'
    r'niacinamide|tranexamic|kojic|arbutin|azelaic|dull|discolor|melasma|fade'
)

def is_antispot(row):
    name = str(row.get('product_name', ''))
    hi   = str(row.get('highlights', ''))
    return bool(
        re.search(ANTISPOT_KEYWORDS, name, re.I) or
        re.search(r'Brightening|Dark Spot|Even Skin Tone|Hyperpigmentation|Vitamin C', hi, re.I)
    )

df['segment'] = 'other_skincare'  # default
df.loc[df['primary_category'] == 'Makeup', 'segment'] = 'makeup'

# Apply anti-spot filter (only to skincare)
skincare_mask = df['primary_category'] == 'Skincare'
antispot_mask = df.apply(is_antispot, axis=1)
df.loc[skincare_mask & antispot_mask, 'segment'] = 'antispot'

print(df['segment'].value_counts())

## 3. Extract Decision Sentences

We look for sentences that contain purchase-trigger language:
- `I bought / chose / picked / switched to / tried this because...`
- `I got this / decided to try / the reason I...`
- `bought this because / chose this after...`

In [ ]:
DECISION_PATTERN = re.compile(
    r'(?:'
    r'I\s+(?:bought|chose|picked|purchased|switched\s+to|tried\s+this|got\s+this|decided\s+to\s+try)'
    r'|bought\s+this|chose\s+this|picked\s+this|switched\s+to\s+this'
    r'|the\s+reason\s+I|decided\s+to\s+try|what\s+made\s+me\s+(?:buy|choose|try|pick)'
    r')[^.!?]{5,250}',
    re.IGNORECASE
)

def extract_decision_sentences(text: str) -> list[str]:
    return DECISION_PATTERN.findall(text)

# Extract (takes ~1 min on full dataset)
df['decision_sents'] = df['review_text'].apply(extract_decision_sentences)
df['has_decision'] = df['decision_sents'].apply(len) > 0

print(f"Reviews with decision sentence: {df['has_decision'].sum():,} ({df['has_decision'].mean()*100:.1f}%)")
print()
print('Decision sentence rate by segment:')
print(df.groupby('segment')['has_decision'].mean().sort_values(ascending=False).map('{:.1%}'.format))

## 4. Define Signal Categories & Keywords

In [ ]:
SIGNALS = {
    'ingredient_driven': re.compile(
        r'niacinamide|vitamin\s*c|tranexamic|retinol|retinoid|kojic|arbutin|azelaic|'
        r'hyaluronic|glycolic|salicylic|lactic|peptide|ceramide|squalane|'
        r'ingredient|formula|chemical|active|concentration|percentage|%|'
        r'spf|sunscreen|antioxidant|aha|bha|pha',
        re.IGNORECASE
    ),
    'brand_driven': re.compile(
        r'love\s+(?:this\s+)?brand|trust\s+(?:this\s+)?brand|loyal|always\s+buy|'
        r'been\s+using\s+(?:this\s+)?brand|go.?to\s+brand|favorite\s+brand|'
        r'only\s+use|swear\s+by|stick\s+with|never\s+disappoint',
        re.IGNORECASE
    ),
    'authority_driven': re.compile(
        r'dermatologist|derm\b|esthetician|aesthetician|doctor|physician|'
        r'recommended\s+by|suggested\s+by|reddit|tiktok|youtube|influencer|'
        r'blogger|review\s+online|saw\s+(?:it\s+)?on|trending|viral|'
        r'skincare\s+community|r/skincare',
        re.IGNORECASE
    ),
    'price_driven': re.compile(
        r'affordable|budget|cheap|inexpensive|dupe|drugstore|'
        r'cheaper\s+than|more\s+affordable|half\s+the\s+price|'
        r'value|worth\s+the\s+(?:price|money|cost)|on\s+sale|'
        r'price\s+point|cost.?effective|bang\s+for',
        re.IGNORECASE
    ),
    'result_driven': re.compile(
        r'before\s+(?:and|&|/)\s*after|results?|before\/after|'
        r"friend(?:'s)?\s+(?:results?|recommendation|skin|told\s+me)"
        r'|my\s+(?:friend|sister|mom|partner)\s+(?:uses?|recommends?|swears?|told)'
        r'|saw\s+(?:someone|a\s+friend|results?|improvement|difference)'
        r'|after\s+seeing|transformation|worked\s+for\s+(?:her|him|them|my)',
        re.IGNORECASE
    ),
}

def classify_signals(sentences: list[str]) -> dict:
    """Return binary flag for each signal category across all decision sentences."""
    combined = ' '.join(sentences)
    return {sig: int(bool(pat.search(combined))) for sig, pat in SIGNALS.items()}

print('Signal categories defined:', list(SIGNALS.keys()))

In [ ]:
# Apply only to reviews that have decision sentences
has_decision = df[df['has_decision']].copy()

signal_df = has_decision['decision_sents'].apply(classify_signals).apply(pd.Series)
has_decision = pd.concat([has_decision.reset_index(drop=True), signal_df.reset_index(drop=True)], axis=1)

# Any signal flag
signal_cols = list(SIGNALS.keys())
has_decision['any_signal'] = has_decision[signal_cols].max(axis=1)

print(f'Reviews with decision sentences: {len(has_decision):,}')
print(f'  → with at least one signal: {has_decision["any_signal"].sum():,}')
print()
print('Overall signal distribution (% of decision-sentence reviews):')
print((has_decision[signal_cols].mean() * 100).sort_values(ascending=False).map('{:.1f}%'.format))

## 5. Compare Signal Distribution: Anti-spot vs. Other Skincare vs. Makeup

In [ ]:
# Mean signal rate per segment
seg_signals = (
    has_decision
    .groupby('segment')[signal_cols]
    .mean()
    .mul(100)
    .round(1)
)

# Reorder rows
seg_signals = seg_signals.reindex(['antispot', 'other_skincare', 'makeup'])
seg_signals.index = ['Anti-spot', 'Other Skincare', 'Makeup']

# Rename columns for display
seg_signals.columns = ['Ingredient', 'Brand', 'Authority\n(derm/social)', 'Price/Value', 'Result/Social Proof']

print('Signal rate by segment (% of decision-sentence reviews):')
print(seg_signals.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Purchase Decision Signals: Anti-spot vs. Other Skincare vs. Makeup', fontsize=13, fontweight='bold', y=1.02)

colors = {'Anti-spot': '#c0392b', 'Other Skincare': '#2980b9', 'Makeup': '#8e44ad'}

# --- Plot 1: grouped bar chart ---
ax1 = axes[0]
x = np.arange(len(seg_signals.columns))
width = 0.25
for i, (seg, color) in enumerate(colors.items()):
    ax1.bar(x + i * width, seg_signals.loc[seg], width, label=seg, color=color, alpha=0.85)

ax1.set_xticks(x + width)
ax1.set_xticklabels(seg_signals.columns, fontsize=9)
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.set_ylabel('% of decision-sentence reviews')
ax1.set_title('Signal Rate by Category')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# --- Plot 2: Anti-spot lift over Makeup (index = Makeup baseline) ---
ax2 = axes[1]
lift = (seg_signals.loc['Anti-spot'] / seg_signals.loc['Makeup'] - 1) * 100
lift_colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in lift]
ax2.barh(seg_signals.columns, lift, color=lift_colors, alpha=0.85)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Anti-spot lift vs. Makeup baseline (%)')
ax2.set_title('Anti-spot Signal Lift vs. Makeup')
ax2.grid(axis='x', alpha=0.3)
for i, v in enumerate(lift):
    ax2.text(v + (1 if v >= 0 else -1), i, f'{v:+.0f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/purchase_signals_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to data/processed/purchase_signals_comparison.png')

## 6. Top Keywords Within Each Signal — Anti-spot Only

In [ ]:
antispot_dec = has_decision[has_decision['segment'] == 'antispot'].copy()

# Specific ingredient keywords in anti-spot decision sentences
INGREDIENT_TERMS = [
    'niacinamide', 'vitamin c', 'tranexamic', 'retinol', 'retinoid', 'kojic',
    'arbutin', 'azelaic', 'glycolic', 'salicylic', 'lactic', 'peptide',
    'ceramide', 'hyaluronic', 'aha', 'bha', 'spf', 'sunscreen', 'squalane'
]
AUTHORITY_TERMS = [
    'dermatologist', 'derm', 'esthetician', 'doctor', 'reddit', 'tiktok',
    'youtube', 'influencer', 'blogger', 'recommended by'
]
PRICE_TERMS = [
    'affordable', 'budget', 'dupe', 'drugstore', 'cheaper', 'value', 'sale',
    'worth the price', 'cost', 'bang for'
]

def count_terms(sentences_list, terms):
    combined = ' '.join(' '.join(s) for s in sentences_list).lower()
    return Counter({t: len(re.findall(re.escape(t), combined)) for t in terms})

ing_counts  = count_terms(antispot_dec['decision_sents'], INGREDIENT_TERMS)
auth_counts = count_terms(antispot_dec['decision_sents'], AUTHORITY_TERMS)
price_counts= count_terms(antispot_dec['decision_sents'], PRICE_TERMS)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Top Keywords in Anti-spot Decision Sentences', fontsize=12, fontweight='bold')

for ax, counts, title, color in zip(
    axes,
    [ing_counts, auth_counts, price_counts],
    ['Ingredient Terms', 'Authority Terms', 'Price/Value Terms'],
    ['#c0392b', '#2980b9', '#27ae60']
):
    top = counts.most_common(10)
    if not top:
        ax.set_title(title + '\n(no matches)')
        continue
    labels, vals = zip(*top)
    ax.barh(list(reversed(labels)), list(reversed(vals)), color=color, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('mention count')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/antispot_keyword_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Hypothesis Test: Is the Difference Statistically Significant?

In [ ]:
from scipy import stats

antispot_rows   = has_decision[has_decision['segment'] == 'antispot']
makeup_rows     = has_decision[has_decision['segment'] == 'makeup']
skincare_rows   = has_decision[has_decision['segment'] == 'other_skincare']

print('=== Hypothesis Tests: Anti-spot vs. Makeup (chi-square) ===\n')
results = []
for sig in signal_cols:
    a_pos = antispot_rows[sig].sum()
    a_neg = len(antispot_rows) - a_pos
    m_pos = makeup_rows[sig].sum()
    m_neg = len(makeup_rows) - m_pos
    chi2, p, _, _ = stats.chi2_contingency([[a_pos, a_neg], [m_pos, m_neg]])
    results.append({
        'signal':          sig,
        'antispot_%':      round(a_pos / len(antispot_rows) * 100, 1),
        'makeup_%':        round(m_pos / len(makeup_rows) * 100, 1),
        'chi2':            round(chi2, 1),
        'p_value':         round(p, 4),
        'significant':     '✓' if p < 0.05 else '✗'
    })

result_df = pd.DataFrame(results).set_index('signal')
print(result_df.to_string())

## 8. Sample Decision Sentences by Signal (Anti-spot)

In [ ]:
for sig in signal_cols:
    sample_rows = antispot_rows[antispot_rows[sig] == 1].head(3)
    print(f'\n── {sig.upper()} ──')
    for _, row in sample_rows.iterrows():
        for sent in row['decision_sents'][:1]:  # one sentence per review
            print(f'  • {sent.strip()[:160]}')

## 9. Export Summary for Power BI

In [ ]:
# Segment-level summary
summary = (
    has_decision
    .groupby('segment')[signal_cols]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)
summary.columns = ['segment'] + [s + '_pct' for s in signal_cols]
summary.to_csv('../data/processed/purchase_signal_summary.csv', index=False)

# Row-level (for drill-down)
export_cols = ['product_id', 'product_name', 'brand_name_review', 'segment',
               'rating', 'price_usd', 'skin_type'] + signal_cols
export_cols = [c for c in export_cols if c in has_decision.columns]
has_decision[export_cols].to_csv('../data/processed/purchase_drivers_rows.csv', index=False)

print('Exported:')
print('  data/processed/purchase_signal_summary.csv')
print('  data/processed/purchase_drivers_rows.csv')
print()
print(summary)